In [16]:
import random
from datetime import datetime
import mysql.connector
from faker import Faker
from pydeequ.analyzers import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [2]:
Faker.seed(42)
fake = Faker(['ko_KR', 'en_US'])

In [3]:
spark = SparkSession.builder.appName("PyDeequ Example") \
    .master("spark://localhost:7077") \
    .config("spark.sql.shuffle.partitions", "1") \
    .getOrCreate()

26/01/06 22:12:04 WARN Utils: Your hostname, MacBook-Pro-14.local resolves to a loopback address: 127.0.0.1; using 10.12.2.175 instead (on interface en0)
26/01/06 22:12:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/06 22:12:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/06 22:12:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
data = [{
    "id": i + 1,
    "name": fake.name(),
    "age": fake.random_int(min=20, max=65),
    "gender": random.choice(['남성', '여성']),
    "address": fake.address(),
    "job": fake.job(),
    "email": fake.email()
} for i in range(100)]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("address", StringType(), True),
    StructField("job", StringType(), True),
    StructField("email", StringType(), True)
])

In [5]:
df = spark.createDataFrame(data=data, schema=schema)

In [6]:
analysis_runner = AnalysisRunner(spark)
analysis_result = (
    analysis_runner.onData(df)
    .addAnalyzer(Size())
    .addAnalyzer(Completeness("id"))
    .addAnalyzer(Completeness("name"))
    .addAnalyzer(Completeness("age"))
    .addAnalyzer(Completeness("gender"))
    .addAnalyzer(Completeness("address"))
    .addAnalyzer(Completeness("job"))
    .addAnalyzer(Completeness("email"))
    .run())

In [17]:
result_df = AnalyzerContext.successMetricsAsDataFrame(spark, analysis_result) \
    .withColumn("run_name", lit("daily_batch")) \
    .withColumn("run_id", lit(f"daily_batch_{datetime.now().strftime('%Y%m%d%H%M%S')}")) \
    .withColumn("logical_datetime", lit(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"))
result_df.printSchema()

root
 |-- entity: string (nullable = true)
 |-- instance: string (nullable = true)
 |-- name: string (nullable = true)
 |-- value: double (nullable = false)
 |-- run_name: string (nullable = false)
 |-- run_id: string (nullable = false)
 |-- logical_datetime: string (nullable = false)



In [18]:
connection = mysql.connector.connect(host="127.0.0.1", user="root", password="root", database="mmix")

In [19]:
with connection.cursor() as cursor:
    insert_query = """
                   INSERT INTO etl_analysis_logs (run_name, run_id, logical_datetime, entity, instance, name, value)
                   VALUES (%(run_name)s, %(run_id)s, %(logical_datetime)s, %(entity)s, %(instance)s, %(name)s, %(value)s)
                   ON DUPLICATE KEY UPDATE value = VALUES(value)
                   """
    cursor.executemany(insert_query, [row.asDict() for row in result_df.collect()])
    connection.commit()